In [ ]:
#@title 🚀 Cell 1: Hardware Environment & Gemini AI Agent Configuration
# Check GPU allocation (Nvidia T4 15-16GB VRAM recommended for Google Colab Free Tier)
!nvidia-smi

import os, sys, psutil

# -------------------------------------------------------------------------
# Google Gemini Vision & AI Director Agent API Key
# -------------------------------------------------------------------------
GEMINI_API_KEY_PRESET = "AQ.Ab8RN6IG8w-7Np_Pra9N_tXsNQ5g4iOACNVbC1ES-jbBcE6ezA"

try:
    from google.colab import userdata
    g_key = userdata.get('GEMINI_API_KEY') or userdata.get('GOOGLE_API_KEY')
    if g_key:
        os.environ['GEMINI_API_KEY'] = g_key
        print("✅ Google Gemini API Key detected from Colab Secrets.")
    else:
        os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY_PRESET
        print("✅ Google Gemini API Key configured from Preset.")
except Exception:
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY_PRESET
    print("✅ Google Gemini API Key configured from Preset.")

os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']

try:
    import torch
    print("=" * 60)
    print("CineFlow-AI: System Diagnostic")
    print("=" * 60)
    print(f"Python Version: {sys.version.split()[0]}")
    print(f"PyTorch Version: {torch.__version__}")
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"Total VRAM: {vram_gb:.2f} GB")
        cc_major, cc_minor = torch.cuda.get_device_capability(0)
        print(f"Compute Capability: {cc_major}.{cc_minor} (T4 CC 7.5 Turing Architecture)")
    else:
        print("⚠️ CUDA is not active. Please navigate to Runtime -> Change runtime type -> T4 GPU.")
except ImportError:
    print("PyTorch not yet imported; will be verified after dependency installation.")

ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System Host RAM: {ram_gb:.2f} GB (Ceiling: ~12.7 GB on Colab Free Tier)")
print("=" * 60)


In [ ]:
#@title 📦 Cell 2: Git Clone & Studio Directory Setup
import os
from pathlib import Path

# Ensure current working directory is inside CineFlow-AI studio root
REPO_NAME = "AI-Video-Studio"
WORKSPACE_DIR = "/content/" + REPO_NAME

if not os.path.exists("modules") and not os.path.exists("app.py"):
    if not os.path.exists(WORKSPACE_DIR):
        print("Cloning CineFlow-AI repository from GitHub...")
        !git clone https://github.com/cineflow-ai/cineflow-ai.git {WORKSPACE_DIR}
    if os.path.exists(WORKSPACE_DIR):
        os.chdir(WORKSPACE_DIR)

print(f"Active Studio Directory: {os.getcwd()}")

# Create all operational pipeline directories
required_dirs = [
    "models",
    "outputs",
    "outputs/masters",
    "outputs/temp",
    "outputs/temp_lipsync",
    "character_profiles",
    "configs",
]
for folder in required_dirs:
    os.makedirs(folder, exist_ok=True)

print("✅ Directory structure initialized successfully.")


In [ ]:
#@title 📥 Cell 3: Install Production Dependencies & FFmpeg
# Install FFmpeg system binary for broadcast-grade H.264 / AAC MP4 multiplexing
!apt-get -y update && apt-get install -y ffmpeg

# Upgrade pip and install all pinned dependencies
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install google-generativeai google-genai

print("✅ All CineFlow-AI requirements installed and verified.")


In [ ]:
#@title 🧠 Cell 4: Download Model Weights & Character Face Bank
import os, urllib.request
from tqdm import tqdm

class DownloadProgressBar(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url: str, output_path: str):
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1024:
        print(f"File already exists: {output_path} ({os.path.getsize(output_path) / (1024*1024):.1f} MB)")
        return
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    print(f"Downloading {os.path.basename(output_path)} from {url}...")
    try:
        with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=os.path.basename(output_path)) as t:
            urllib.request.urlretrieve(url, filename=output_path, reporthook=t.update_to)
    except Exception as e:
        print(f"Note: Download of {os.path.basename(output_path)} failed: {e}. Pipeline will use high-order procedural fallback.")

# Checkpoint download registry
MODEL_REGISTRY = {
    "models/RealESRGAN_x4plus.pth": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth",
}

for dest_path, url in MODEL_REGISTRY.items():
    download_file(url, dest_path)

print("✅ Model checkpoints downloaded and Face Bank verified.")


In [ ]:
#@title 🧪 Cell 5: Automated Test Suite Verification (pytest)
# Executes complete diagnostic test suite verifying VRAMManager, CharacterStudio, CineVideoEngine, LipSyncEngine, PostProductionEngine
!pytest tests/test_pipeline_e2e.py tests/test_character_engine.py tests/test_universal_agent.py -v --tb=short


In [ ]:
#@title 🎬 Cell 6: Launch CineFlow-AI Studio WebUI (Public Share Link)
# Starts the Beginner-Friendly Gradio Studio with public URL (share=True) for Google Colab remote browser access
!python app.py --share --port 7860 --config configs/colab_t4_config.yaml
